# Step 9 — Post-Round Debrief Engine
**Tough Talks · Phase 4**

Goal: prove Gemma 4 E2B (text-only) can score a finished practice round and return specific, schema-conforming coaching feedback that names what the user said, why it lost ground, and what to say next time.

The output is one structured JSON object matching `data/schemas/debrief.schema.json`:

- `ground_lost[]` — each entry `{turn, quote (verbatim), reason}`. Turns where the user gave up leverage.
- `over_apologies[]` — each entry `{turn, quote}`. User turns containing unnecessary apology cues.
- `missed_openings[]` — each entry `{turn, description, better_line}`. The `better_line` is a ready-to-say sentence in the USER's voice.
- `wins[]` — each entry `{turn, description}`. What worked and should be repeated.
- `one_fix_next_time` — a single sentence, the most important change.

**Architecture** — same one-shot hybrid-runtime pattern as Step 8:
- Runtime in `backend/core/_runtime/debrief.py`. Notebook is a thin driver.
- Single prompt-based JSON call (analytical, not multi-turn).
- Two-shot retry: greedy first, then a single light-sampling pass (`temperature=0.3, top_p=0.9, top_k=64`) on JSON / validation failure.
- System-contract enforcement in code: `turn` indices clamped to `[1, user_turn_count]`, malformed array entries dropped silently (one bad `ground_lost` entry should not lose three valid `wins`), required string fields rejected when empty, `one_fix_next_time` required and non-empty.
- `enable_thinking` is exposed as a knob, defaulting to `False`. The final cell A/B-compares thinking-on vs thinking-off. Debrief is the **second** analytical task in the Phase 4 hypothesis test — it writes content for multiple speaker roles in one payload (verbatim USER quotes, descriptions of opponent moves, `better_line` in the USER's voice) which is exactly the shape thinking-on helped on premortem. N=3 datapoint to promote / demote `[[hypothesis-persona-thinking-helps]]`.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `generate_debrief()` returns a schema-conforming dict on the hardcoded Step-7 transcript.
3. Every `turn` value is an integer in `[1, 5]` (the practice round has 5 user turns).
4. Every `quote` in `ground_lost` / `over_apologies` is a verbatim substring of the user turn it references.
5. Every required string field is non-empty.
6. `one_fix_next_time` is a single sentence grounded in something concrete from the transcript.
7. The A/B cell renders both `enable_thinking=False` and `enable_thinking=True` outputs side-by-side.

**Note on inputs.** Both Jamie's PersonVault profile, the user's TalkDNA, and the 5-turn practice transcript are hardcoded inline (matching Step 06 v2 / Step 05 v2 / Step 07 Run-2 outputs respectively) so this notebook doesn't re-pay Steps 05 / 06 / 07's combined ~30-minute LLM cost on every run. In production these come from `analyze_person_vault()`, `analyze_talk_dna()`, and `run_practice_conversation()`.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────────────────
# Text-only path — no audio libs required. Same rule as Steps 05 / 06 /
# 07 / 08: bump only transformers + accelerate on Colab / Kaggle
# (bumping torch breaks the pre-installed torchvision / CUDA pairing).
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers — the already-imported version won't
# pick up the change.

!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 104.8 MB/s eta 0:00:0000:01:01


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────
# Same shim as Steps 01–08 — auto-clones / refreshes on Colab / Kaggle
# and clears any cached `backend.*` modules so the imports below pick
# up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ──────────────────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    DEFAULT_MODEL_ID,
    DebriefConfig,
    DebriefError,
    LoadConfig,
    count_user_turns,
    format_persona_profile_block,
    format_practice_transcript,
    format_talk_dna_block,
    generate_debrief,
    load_model,
)

In [4]:
# ── 3. Configuration ───────────────────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "debrief.schema.json"

# PersonVault profile for Jamie — same v2 dict as Steps 07 / 08.
# Inlined so this notebook doesn't re-pay Step 06's ~15-minute LLM cost
# on every run. In production this comes from `analyze_person_vault()`.
JAMIE_PROFILE = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "suggesting escalation when blockers are still open",
            "implying missed communication is one-sided",
            "setting hard deadlines without acknowledging blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
            "acknowledging the tightness of the timeline",
            "proposing a concrete next-step the user will own",
        ],
        "common_deflections": [
            "I told you",
            "Don't blame me",
            "Don't put this on me",
            "You weren't there",
            "staging tables aren't done",
        ],
        "responds_best_to": (
            "Clear, actionable next steps tied to specific milestones. "
            "Responds well when the user accepts responsibility for "
            "communication gaps."
        ),
    },
}

# Representative TalkDNA profile for the user — matches the v2 shape
# Step 05 produces (apology-prone, hedges, silence_under_pressure
# true). Inlined for the same reason as Jamie's profile.
USER_TALK_DNA = {
    "user_id": "local",
    "version": 2,
    "conversation_count": 2,
    "patterns": {
        "filler_phrases": ["I just feel like", "kind of", "sort of"],
        "apology_rate": 0.57,
        "silence_under_pressure": True,
        "sarcasm_frequency": "low",
        "escalation_triggers": [
            "feeling dismissed",
            "being told to wait without a reason",
        ],
        "avg_turn_length_words": 15.7,
    },
    "strengths": [
        "clear_problem_statement",
        "acknowledges_mistakes",
    ],
    "weaknesses": [
        "over_apologizes",
        "hedges_before_vulnerable_statements",
    ],
    "updated_at": "2026-05-14T00:00:00Z",
}

# What the user was trying to achieve in this practice round — same
# goal that drove Step 07's simulator and Step 08's pre-mortem.
USER_GOAL = (
    "Get Jamie to commit to delivering the staging-tables data by "
    "Wednesday EOD and to acknowledge that the escalation last week "
    "needed to land more clearly — not just be sent."
)

# Five-turn practice transcript, lifted verbatim from Step 07's Run-2
# results table (notebook cell `step07-results`). Persona turns carry
# the exact `resistance_type` / `escalation_level` Step 07 emitted so
# the debrief sees the same arc data the persona simulator did.
PRACTICE_TRANSCRIPT = [
    {
        "speaker": "user",
        "text": (
            "Hey Jamie, I want to talk about what happened with the report "
            "last week. The fallout from missing that deadline is still on "
            "us, and I'd like to make sure it doesn't happen again."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I told you we were short on time. It's not a personal failing, "
            "it's just how things run. What exactly do you want me to do "
            "differently this time?"
        ),
        "resistance_type": "deflect",
        "escalation_level": 0.60,
    },
    {
        "speaker": "user",
        "text": (
            "I hear that the data team hadn't delivered, but the report "
            "date matters more than who knew what on which day. Can we "
            "agree the staging data lands by Wednesday EOD this time?"
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "Wednesday EOD is tight. I can try to push it, but I need to "
            "know exactly what blockers you're anticipating so I can manage "
            "expectations properly. Don't just set a deadline without "
            "outlining the path to get there."
        ),
        "resistance_type": "counter_attack",
        "escalation_level": 0.75,
    },
    {
        "speaker": "user",
        "text": (
            "I'm not saying you didn't escalate. I'm saying the escalation "
            "has to land — if I miss your emails, that's still our problem "
            "to solve together, not yours alone."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I told you, don't put this on me. If the communication isn't "
            "clear, that's on the process, not just me. I'll look at the "
            "timeline again, but I can't guarantee Wednesday EOD without a "
            "concrete commitment from your side on what you can handle."
        ),
        "resistance_type": "guilt_trip",
        "escalation_level": 0.70,
    },
    {
        "speaker": "user",
        "text": (
            "What if we set up a five-minute Slack check-in each morning "
            "until the staging tables are done? You flag what's blocked, I "
            "unblock it that day. That way nothing has to escalate at all."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "A daily check-in sounds like a lot of overhead right now. If "
            "you're going to own the unblocking, then fine. Send me the "
            "specific blockers you see by the end of today, and I'll see "
            "what I can manage on my end."
        ),
        "resistance_type": "deflect",
        "escalation_level": 0.50,
    },
    {
        "speaker": "user",
        "text": (
            "Thank you, that means a lot. And next time something blocks "
            "the report, can you ping me directly instead of email? I'll "
            "do the same when I'm the one missing things."
        ),
    },
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": (
            "I can do that. Just make sure those pings are actionable, not "
            "just general updates. I'll keep an eye out for them."
        ),
        "resistance_type": "concede",
        "escalation_level": 0.40,
    },
]

N_USER_TURNS = count_user_turns(PRACTICE_TRANSCRIPT)

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Persona            : {JAMIE_PROFILE['name']} ({JAMIE_PROFILE['relationship_type']})")
print(f"Transcript turns   : {len(PRACTICE_TRANSCRIPT)} total / {N_USER_TURNS} USER")

Model              : google/gemma-4-E2B-it
Device             : cuda
Persona            : Jamie (colleague)
Transcript turns   : 10 total / 5 USER


In [5]:
# ── 4. Preview the rendered context blocks (no model needed) ──────────────
# All three rendered blocks are what the runtime feeds into the prompt.
# Rendering them here is purely diagnostic — it lets us verify the
# inputs are well-formed before paying for the model load.

print("=" * 76)
print("PERSON PROFILE BLOCK (Jamie)")
print("=" * 76)
print(format_persona_profile_block(JAMIE_PROFILE))

print()
print("=" * 76)
print("TALK DNA BLOCK (user)")
print("=" * 76)
print(format_talk_dna_block(USER_TALK_DNA))

print()
print("=" * 76)
print("PRACTICE TRANSCRIPT (5 user turns, 5 Jamie turns)")
print("=" * 76)
print(format_practice_transcript(PRACTICE_TRANSCRIPT))

PERSON PROFILE BLOCK (Jamie)
- communication_style: defensive
- emotional_triggers (USER-side cues that escalate / make you defensive): ['citing past commitments', 'suggesting escalation when blockers are still open', 'implying missed communication is one-sided', 'setting hard deadlines without acknowledging blockers']
- de_escalation_keys (USER-side moves that calm / open you up): ['explicitly disowning blame', 'reframing as joint problem-solving', 'acknowledging the tightness of the timeline', 'proposing a concrete next-step the user will own']
- common_deflections (YOUR habitual evasion phrases): ['I told you', "Don't blame me", "Don't put this on me", "You weren't there", "staging tables aren't done"]
- responds_best_to: Clear, actionable next steps tied to specific milestones. Responds well when the user accepts responsibility for communication gaps.

TALK DNA BLOCK (user)
- weaknesses (USER habits the opponent could exploit): ['over_apologizes', 'hedges_before_vulnerable_statemen

In [6]:
# ── 5. Load the text-only processor + model ────────────────────────
# Debrief reasons over WORDS — same rationale as TalkDNA / PersonVault /
# persona-sim / premortem. The `multimodal=False` (default) path loads
# `AutoModelForCausalLM`, which is lighter on VRAM and slightly faster
# than the multimodal class. Live Mode (Phase 5+) will reuse this same
# loaded model across components, so the load cost is amortised.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [7]:
# ── 6. Generate the debrief (greedy, thinking=False) ─────────────────
# Single prompt-based JSON call — no rolling history (the opposite of
# Step 7's persona simulator). The runtime threads the user goal,
# person profile, TalkDNA, and rendered transcript into the
# `data/prompts/debrief.md` template, calls `chat()` greedy, parses
# the JSON, validates / coerces against the schema, and returns a
# schema-conforming dict.

cfg = DebriefConfig(
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=False,
)

try:
    debrief = generate_debrief(processor, model, cfg=cfg)
except DebriefError as exc:
    print(f"FAILED: {exc}")
    for attempt in getattr(exc, "attempts", []):
        print(f"  attempt ({attempt['sampling']}): {attempt['error']}")
        print(f"  raw_head: {attempt['raw_head']!r}")
    raise

# Render the result inline so the structure is visible field-by-field,
# not just in the validation table below.
print(f"ONE FIX NEXT TIME: {debrief['one_fix_next_time']}\n")

for section in ("ground_lost", "over_apologies", "missed_openings", "wins"):
    items = debrief[section]
    print(f"--- {section}  ({len(items)} item(s)) ---")
    if not items:
        print("  (none)")
    for entry in items:
        turn = entry["turn"]
        if section == "ground_lost":
            print(f"  [USER {turn}] quote : {entry['quote']!r}")
            print(f"          reason: {entry['reason']}")
        elif section == "over_apologies":
            print(f"  [USER {turn}] quote : {entry['quote']!r}")
        elif section == "missed_openings":
            print(f"  [USER {turn}] {entry['description']}")
            print(f"          better_line: {entry['better_line']!r}")
        else:  # wins
            print(f"  [USER {turn}] {entry['description']}")
    print()

ONE FIX NEXT TIME: When addressing past issues, immediately pivot from assigning blame to proposing a joint solution or a concrete, measurable commitment for the future.

--- ground_lost  (2 item(s)) ---
  [USER 3] quote : "I'm not saying you didn't escalate. I'm saying the escalation has to land — if I miss your emails, that's still our problem to solve together, not yours alone."
          reason: The user invited a defensive reaction by framing the issue as 'our problem to solve together' while simultaneously implying Jamie's failure to respond was a personal failing, which triggered Jamie's 'guilt_trip' resistance.
  [USER 3] quote : "I'm not saying you didn't escalate."
          reason: This was a hedge before a vulnerable statement, which aligns with the user's Talk DNA weakness, allowing Jamie to dismiss the core point about the escalation clarity.

--- over_apologies  (1 item(s)) ---
  [USER 5] quote : 'Thank you, that means a lot.'

--- missed_openings  (2 item(s)) ---
  [USE

In [8]:
# ── 7. Schema validation + results table ────────────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05 / 06
# / 07 / 08). Per array:
#   - required schema fields present on every entry
#   - turn integer in [1, N_USER_TURNS]
#   - string fields are non-empty strings
# Cross-debrief:
#   - all 5 required top-level keys present
#   - `one_fix_next_time` is a non-empty string
#   - every `quote` in ground_lost / over_apologies is a verbatim
#     substring of the user turn it references (the prompt's hard
#     rule — if this fails, the model paraphrased)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
TOP_REQUIRED = set(schema["required"])

USER_TEXT_BY_TURN: dict[int, str] = {}
_idx = 0
for entry in PRACTICE_TRANSCRIPT:
    if entry.get("speaker") == "user":
        _idx += 1
        USER_TEXT_BY_TURN[_idx] = entry.get("text") or ""


def _validate_turn_range(turn) -> bool:
    return isinstance(turn, int) and 1 <= turn <= N_USER_TURNS


def _validate_non_empty_str(value) -> bool:
    return isinstance(value, str) and bool(value.strip())


def _validate_quote_verbatim(turn, quote) -> bool:
    """True iff `quote` is a verbatim substring of the user turn it
    references. The prompt requires this; the runtime can't enforce it
    in code without rejecting potentially-useful entries (model could
    quote a slightly normalised form), so the check belongs in the
    results table instead."""
    if not isinstance(quote, str) or not _validate_turn_range(turn):
        return False
    return quote.strip() in USER_TEXT_BY_TURN.get(turn, "")


section_errors: dict[str, list[str]] = {}
for section, required_keys in {
    "ground_lost":     {"turn", "quote", "reason"},
    "over_apologies":  {"turn", "quote"},
    "missed_openings": {"turn", "description", "better_line"},
    "wins":            {"turn", "description"},
}.items():
    errs: list[str] = []
    for i, entry in enumerate(debrief.get(section, []), start=1):
        missing = required_keys - set(entry.keys())
        if missing:
            errs.append(f"entry {i}: missing keys {sorted(missing)}")
        if not _validate_turn_range(entry.get("turn")):
            errs.append(f"entry {i}: turn {entry.get('turn')!r} not in [1, {N_USER_TURNS}]")
        for k in required_keys - {"turn"}:
            if not _validate_non_empty_str(entry.get(k)):
                errs.append(f"entry {i}: {k!r} empty or non-string")
    section_errors[section] = errs

all_quotes_verbatim = all(
    _validate_quote_verbatim(e["turn"], e["quote"])
    for e in debrief.get("ground_lost", []) + debrief.get("over_apologies", [])
)

quote_count = (
    len(debrief.get("ground_lost", []))
    + len(debrief.get("over_apologies", []))
)

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",        True,                                                f"{n_params:.1f}B params on {model.device}"),
    ("all_required_keys_present", TOP_REQUIRED.issubset(debrief.keys()),                f"{sorted(TOP_REQUIRED & set(debrief.keys()))}"),
    ("one_fix_non_empty",        _validate_non_empty_str(debrief.get("one_fix_next_time")), debrief.get("one_fix_next_time", "")[:80] + "..."),
    ("ground_lost_clean",        not section_errors["ground_lost"],                    f"{len(debrief['ground_lost'])} entries"),
    ("over_apologies_clean",     not section_errors["over_apologies"],                 f"{len(debrief['over_apologies'])} entries"),
    ("missed_openings_clean",    not section_errors["missed_openings"],                f"{len(debrief['missed_openings'])} entries"),
    ("wins_clean",               not section_errors["wins"],                           f"{len(debrief['wins'])} entries"),
    ("quotes_are_verbatim",      all_quotes_verbatim if quote_count else True,         f"{quote_count} quoted entries checked"),
    ("debrief_id_attached",      isinstance(debrief.get("debrief_id"), str)
                                  and debrief["debrief_id"].startswith("debrief_"),
                                                                                       f"{debrief.get('debrief_id', '<missing>')}"),
]

print("=" * 76)
print("STEP 9 RESULTS — Gemma 4 Post-Round Debrief Engine")
print("=" * 76)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

for section, errs in section_errors.items():
    if not errs:
        continue
    print(f"\n{section} errors:")
    for line in errs:
        print(f"  - {line}")

print()
print("OVERALL:", "READY FOR STEP 10" if all_ok else "FIX FAILURES ABOVE")

STEP 9 RESULTS — Gemma 4 Post-Round Debrief Engine
[PASS]  text_model_loaded             5.1B params on cuda:0
[PASS]  all_required_keys_present     ['ground_lost', 'missed_openings', 'one_fix_next_time', 'over_apologies', 'wins']
[PASS]  one_fix_non_empty             When addressing past issues, immediately pivot from assigning blame to proposing...
[PASS]  ground_lost_clean             2 entries
[PASS]  over_apologies_clean          1 entries
[PASS]  missed_openings_clean         2 entries
[PASS]  wins_clean                    3 entries
[PASS]  quotes_are_verbatim           3 quoted entries checked
[PASS]  debrief_id_attached           debrief_869cf8c54ee8

OVERALL: READY FOR STEP 10


In [9]:
# ── 8. A/B: enable_thinking=False vs enable_thinking=True (same transcript) ──
# Tests the open hypothesis [[hypothesis-persona-thinking-helps]] on a
# THIRD task family. Debrief is analytical (no character to wash out)
# AND emits content for multiple speaker roles in one payload — verbatim
# USER quotes, descriptions of opponent moves, `better_line` in the
# user's voice. Same shape that thinking-on helped on premortem.
#
# Premortem learning: budget at least 2x the JSON body for the thinking
# branch on analytical tasks. Debrief JSON body is comparable to
# premortem (~600–800 tokens with 3–4 entries per array), so bumping
# the thinking branch to `max_new_tokens=2048` is the same insurance.
#
# Wraps the thinking call in try/except so a future failure prints the
# raw model output (attached to `exc.attempts`) without needing a re-run.

cfg_no_thinking = DebriefConfig(
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=False,
)
cfg_thinking = DebriefConfig(
    transcript=PRACTICE_TRANSCRIPT,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=True,
    max_new_tokens=2048,
)

deb_no_thinking = generate_debrief(processor, model, cfg=cfg_no_thinking)

deb_thinking = None
thinking_err = None
try:
    deb_thinking = generate_debrief(processor, model, cfg=cfg_thinking)
except DebriefError as exc:
    thinking_err = exc


def _render_short(label: str, deb: dict) -> None:
    print(f"\n--- {label} ---")
    print(f"  one_fix_next_time: {deb['one_fix_next_time']}")
    counts = (
        f"ground_lost={len(deb['ground_lost'])}, "
        f"over_apologies={len(deb['over_apologies'])}, "
        f"missed_openings={len(deb['missed_openings'])}, "
        f"wins={len(deb['wins'])}"
    )
    print(f"  counts           : {counts}")
    for section, key in (
        ("ground_lost",     "reason"),
        ("over_apologies",  None),
        ("missed_openings", "better_line"),
        ("wins",            "description"),
    ):
        for entry in deb[section]:
            head = f"  [{section}#turn={entry['turn']}]"
            if section in ("ground_lost", "over_apologies"):
                head += f" quote={entry['quote']!r}"
            if key and key in entry:
                head += f" {key}={entry[key]!r}"
            print(head)


print("=" * 76)
print("A/B — enable_thinking on the same practice transcript")
print("=" * 76)

_render_short("thinking=False (default)", deb_no_thinking)

if deb_thinking is not None:
    _render_short("thinking=True", deb_thinking)
else:
    print("\n--- thinking=True ---")
    print(f"  FAILED: {thinking_err}")
    for attempt in getattr(thinking_err, "attempts", []):
        print(f"\n  attempt ({attempt['sampling']}): {attempt['error']}")
        print("  raw_head (first 500 chars):")
        print(f"  {attempt['raw_head']!r}")

A/B — enable_thinking on the same practice transcript

--- thinking=False (default) ---
  one_fix_next_time: When addressing past issues, immediately pivot from assigning blame to proposing a joint solution or a concrete, measurable commitment for the future.
  counts           : ground_lost=2, over_apologies=1, missed_openings=2, wins=3
  [ground_lost#turn=3] quote="I'm not saying you didn't escalate. I'm saying the escalation has to land — if I miss your emails, that's still our problem to solve together, not yours alone." reason="The user invited a defensive reaction by framing the issue as 'our problem to solve together' while simultaneously implying Jamie's failure to respond was a personal failing, which triggered Jamie's 'guilt_trip' resistance."
  [ground_lost#turn=3] quote="I'm not saying you didn't escalate." reason="This was a hedge before a vulnerable statement, which aligns with the user's Talk DNA weakness, allowing Jamie to dismiss the core point about the escalation cla

In [10]:
# ── 9. Final debrief dict — the JSON Phase 5's API will return verbatim ────
# Step 10 (Aftermath — plan vs reality) will consume this dict to
# compare the pre-mortem's predicted failure scenarios against what
# actually happened in the round. Prefer the thinking=True debrief
# when both branches succeeded (premortem's N=2 finding was that
# thinking-on caught cross-field inconsistencies that thinking-off
# didn't on analytical payloads); fall back to thinking=False
# otherwise. Contract test in cell 7 already validated the structure;
# this cell is the demo handoff.

_source_debrief = (
    deb_thinking
    if ("deb_thinking" in globals() and deb_thinking is not None)
    else debrief
)
_source_label = (
    "cell-8 thinking=True"
    if ("deb_thinking" in globals() and deb_thinking is not None)
    else "cell-6 thinking=False"
)

print("=" * 76)
print(f"FINAL DEBRIEF (source: {_source_label})")
print("=" * 76)
print(json.dumps(_source_debrief, indent=2, ensure_ascii=False))

FINAL DEBRIEF (source: cell-8 thinking=True)
{
  "ground_lost": [
    {
      "turn": 3,
      "quote": "I'm not saying you didn't escalate.",
      "reason": "The user attempted to re-anchor the conversation on shared responsibility, but Jamie immediately used a guilt-trip deflection ('I told you, don't put this on me') to shift the focus back to personal blame rather than accepting the joint problem-solving framing."
    }
  ],
  "over_apologies": [
    {
      "turn": 5,
      "quote": "Thank you, that means a lot."
    }
  ],
  "missed_openings": [
    {
      "turn": 1,
      "description": "The user could have immediately used a de-escalation key by explicitly disowning blame regarding the past failure, rather than starting with 'The fallout from missing that deadline is still on us.'",
      "better_line": "I take responsibility for the missed deadline last week, and I want to make sure we fix the process so it doesn't happen again."
    }
  ],
  "wins": [
    {
      "turn": 2,